# Qwen3.5-4B-Hmm Q4_K_M GGUF → CPU ONNX with Microsoft ONNXRuntime Mobius (Fixed)

This notebook will:

1. Check out a pinned revision of `onnxruntime/mobius` from GitHub, build a wheel from source, and install it instead of using a prebuilt Mobius wheel from PyPI.
2. Download and SHA-256 verify `Qwen3.5-4B-Hmm-Q4_K_M.gguf` from Hugging Face.
3. Convert it with `mobius build-gguf ... --ep cpu --dtype f32` and save the output to `/content/onnx_outputs`.
4. Validate the ONNX model and quantization report, select the CPU Execution Provider, and run stateful Qwen3.5 CPU inference.
5. Implement `noul`, `choice`, and `score` decision APIs using the exact Hmm prompt and first-token option-letter probabilities described by the model card.

> A Colab High-RAM CPU runtime is recommended. The source GGUF is approximately 2.71 GB, and conversion, external weights, and ORT session creation require additional disk space and RAM.

> The Mobius capability catalog marks `qwen35` config, tensor mapping, graph construction, and quantized import as supported, while real-weight runtime evidence remains pending. The direct ORT tests below are therefore required acceptance checks. The notebook fails if session creation or inference fails; producing ONNX files alone is not treated as success.


In [ ]:
from pathlib import Path
import os

HF_REPO = "n4ze3m/Qwen3.5-4B-Hmm"
HF_REVISION = "c27fa3c627dfaced343c6ba9d3a0d00243a3be51"
GGUF_FILENAME = "Qwen3.5-4B-Hmm-Q4_K_M.gguf"
EXPECTED_SHA256 = "5e03cb057049c56b421bd3c506d77fd8e0a77996bb8464148a02cfc3caac5229"
TOKENIZER_REPO = "Qwen/Qwen3.5-4B"
TOKENIZER_REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"

# Pinned Mobius source revision used when this notebook was authored.
MOBIUS_REPO = "https://github.com/onnxruntime/mobius.git"
MOBIUS_REVISION = "6b27a3f08b8b5d08ba9b14b416e3b435942bb0bd"

WORKDIR = (
    Path("/content")
    if "COLAB_RELEASE_TAG" in os.environ
    else Path.cwd() / ".mobius_colab_run"
)
MOBIUS_DIR = WORKDIR / "mobius"
DIST_DIR = WORKDIR / "mobius_dist"
DOWNLOAD_DIR = WORKDIR / "gguf_source"
OUTPUT_DIR = WORKDIR / "onnx_outputs"

# User-requested Microsoft package feed.
os.environ["PIP_INDEX_URL"] = "https://packagefeedproxy.microsoft.io/pypi/simple/"
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print({
    "gguf": f"{HF_REPO}@{HF_REVISION}:{GGUF_FILENAME}",
    "mobius_revision": MOBIUS_REVISION,
    "output": str(OUTPUT_DIR),
    "pip_index": os.environ["PIP_INDEX_URL"],
})


## 1. Environment and capacity checks

Keep at least 12 GiB of free disk space. Using `--dequantize --dtype f32` requires substantially more space and is not recommended on a standard Colab runtime.


In [ ]:
import platform
import shutil
import sys

WORKDIR.mkdir(parents=True, exist_ok=True)
total, used, free = shutil.disk_usage(WORKDIR)
free_gib = free / 1024**3
print("Python:", sys.version)
print("Platform:", platform.platform())
print(f"Free disk: {free_gib:.2f} GiB")
assert sys.version_info >= (3, 10), "Mobius requires Python >= 3.10"
assert free_gib >= 12, "At least 12 GiB of free disk space is required; use a larger Colab runtime or clean /content"


## 2. Build and install Mobius from source

This step checks out a pinned commit, runs `python -m build --wheel`, and installs the newly built wheel. You may change `MOBIUS_REVISION` to test a newer revision, but the result will no longer correspond to this notebook's pinned version.


In [ ]:
import subprocess

def run(cmd, *, cwd=None):
    print("+", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)

if not MOBIUS_DIR.exists():
    run(["git", "clone", "--filter=blob:none", MOBIUS_REPO, MOBIUS_DIR])
run(["git", "fetch", "--depth", "1", "origin", MOBIUS_REVISION], cwd=MOBIUS_DIR)
run(["git", "checkout", "--detach", MOBIUS_REVISION], cwd=MOBIUS_DIR)

# Colab Python 3.13 is marked as an externally managed environment. Do not replace
# its apt-managed pip; install packages explicitly with pip's supported override.
PIP_FLAGS = ["--break-system-packages", "--no-cache-dir"]
run([
    sys.executable, "-m", "pip", "install", *PIP_FLAGS,
    "setuptools>=77", "wheel", "build",
])
DIST_DIR.mkdir(parents=True, exist_ok=True)
run([
    sys.executable, "-m", "build", "--wheel", "--no-isolation",
    "--outdir", DIST_DIR,
], cwd=MOBIUS_DIR)

wheels = sorted(DIST_DIR.glob("mobius_onnx-*.whl"))
assert len(wheels) == 1, f"Expected one Mobius wheel, found: {wheels}"
run([
    sys.executable, "-m", "pip", "install", *PIP_FLAGS, "--upgrade",
    str(wheels[0]), "gguf>=0.10.0", "onnxruntime>=1.28.0", "onnx", "huggingface_hub"
])


In [ ]:
# If Colab reports an import ABI error after installing or upgrading binary packages,
# select Runtime > Restart session and continue from this cell. A restart is normally unnecessary.
import importlib.metadata as metadata
import onnxruntime as ort

installed_mobius = metadata.version("mobius-onnx")
installed_ort = metadata.version("onnxruntime")
checked_out = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=MOBIUS_DIR, text=True
).strip()

print("mobius-onnx:", installed_mobius)
print("Mobius source commit:", checked_out)
print("onnxruntime:", installed_ort)
print("ORT providers:", ort.get_available_providers())
assert checked_out == MOBIUS_REVISION
assert "CPUExecutionProvider" in ort.get_available_providers()


## 3. Download and verify the specified GGUF

Download the pinned Hugging Face commit and verify its Hub metadata, file size, and SHA-256 so model updates or download corruption cannot silently make the result unreproducible.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import hashlib

api = HfApi()
info = api.model_info(HF_REPO, revision=HF_REVISION, files_metadata=True)
remote = next((s for s in info.siblings if s.rfilename == GGUF_FILENAME), None)
assert remote is not None, f"{GGUF_FILENAME} not found in {HF_REPO}@{HF_REVISION}"

remote_size = remote.size
remote_sha = getattr(getattr(remote, "lfs", None), "sha256", None)
print("Hub commit:", info.sha)
print("Remote size:", remote_size)
print("Remote LFS SHA-256:", remote_sha)
assert info.sha == HF_REVISION
if remote_sha is not None:
    assert remote_sha == EXPECTED_SHA256

gguf_path = Path(hf_hub_download(
    repo_id=HF_REPO,
    filename=GGUF_FILENAME,
    revision=HF_REVISION,
    local_dir=DOWNLOAD_DIR,
))

def sha256_file(path: Path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

actual_sha = sha256_file(gguf_path)
print("Downloaded:", gguf_path)
print("Local size:", gguf_path.stat().st_size)
print("Local SHA-256:", actual_sha)
assert remote_size is None or gguf_path.stat().st_size == remote_size
assert actual_sha == EXPECTED_SHA256


In [ ]:
from gguf import GGUFReader

reader = GGUFReader(str(gguf_path), mode="r")

def field_scalar(name):
    field = reader.fields.get(name)
    if field is None:
        return None
    value = field.parts[field.data[0]]
    if hasattr(value, "tobytes"):
        raw = value.tobytes()
        try:
            return raw.decode("utf-8")
        except UnicodeDecodeError:
            pass
    if hasattr(value, "item") and getattr(value, "size", 0) == 1:
        value = value.item()
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    return value

architecture = field_scalar("general.architecture")
print("GGUF architecture:", architecture)
print("GGUF tensors:", len(reader.tensors))
assert architecture == "qwen35", f"Expected qwen35, got {architecture!r}"
del reader


## 4. Convert into `onnx_outputs/` with `mobius build-gguf`

- `--ep cpu`: build and optimize the graph for the ONNX Runtime CPU execution provider.
- `--dtype f32`: use FP32 for floating-point CPU operations.
- No `--dequantize`: preserve quantized storage where Mobius supports it instead of expanding the 4B model into full FP32 weights.
- `--release`: remove build-time debug and provenance metadata without changing the graph or inference behavior.

Mobius may convert the Q4_K source into packed INT4 affine storage. This can be a lossy repack; consult `quantization_report.json` for the exact disposition.


In [ ]:
import shutil

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

run([
    "mobius", "build-gguf", str(gguf_path),
    "--output", str(OUTPUT_DIR),
    "--ep", "cpu",
    "--dtype", "f32",
    "--release",
])

assert OUTPUT_DIR.is_dir()
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(OUTPUT_DIR)}\t{path.stat().st_size / 1024**2:.2f} MiB")


## 5. Static validation and quantization report


In [ ]:
import json
import onnx

model_candidates = sorted(OUTPUT_DIR.rglob("*.onnx"))
assert len(model_candidates) == 1, f"Expected one ONNX model, found {model_candidates}"
model_path = model_candidates[0]
quant_report_path = OUTPUT_DIR / "quantization_report.json"
assert model_path.is_file()
assert quant_report_path.is_file()

# Path-based checker supports external-data models without manually merging files.
onnx.checker.check_model(str(model_path), full_check=False)
print("ONNX checker: PASS")

quant_report = json.loads(quant_report_path.read_text())
print(json.dumps(quant_report, indent=2)[:12000])

report_text = json.dumps(quant_report).lower()
assert "q4_k" in report_text, "quantization report does not mention source Q4_K"
assert "quant" in report_text, "quantization report lacks quantized-storage evidence"


## 6. CPU session creation and two-step stateful inference

This smoke test does not depend on ORT GenAI or an external tokenizer. It creates the initial Qwen3.5 hybrid DeltaNet/KV state directly from the ONNX I/O contract, runs token 1, feeds each `present.*` state back into `past_key_values.*`, and then runs token 2.


In [ ]:
import numpy as np
import time

session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session_options.enable_mem_pattern = False
session_options.intra_op_num_threads = max(1, (os.cpu_count() or 2) // 2)

started = time.perf_counter()
session = ort.InferenceSession(
    str(model_path),
    sess_options=session_options,
    providers=["CPUExecutionProvider"],
)
print(f"Session created in {time.perf_counter() - started:.2f}s")
print("Active providers:", session.get_providers())
assert session.get_providers()[0] == "CPUExecutionProvider"

for item in session.get_inputs():
    print("INPUT ", item.name, item.type, item.shape)
for item in session.get_outputs():
    print("OUTPUT", item.name, item.type, item.shape)


In [ ]:
ORT_TO_NUMPY = {
    "tensor(float)": np.float32,
    "tensor(float16)": np.float16,
    "tensor(double)": np.float64,
    "tensor(int64)": np.int64,
    "tensor(int32)": np.int32,
    "tensor(bool)": np.bool_,
}

def concrete_state_shape(inp):
    dims = []
    for axis, dim in enumerate(inp.shape):
        if isinstance(dim, int):
            dims.append(dim)
        elif (inp.name.endswith(".key") or inp.name.endswith(".value")) and axis == 2:
            dims.append(0)
        else:
            dims.append(1)
    return tuple(dims)

def initial_states():
    states = {}
    for inp in session.get_inputs():
        if not inp.name.startswith("past_key_values."):
            continue
        dtype = ORT_TO_NUMPY.get(inp.type)
        assert dtype is not None, f"Unsupported state dtype: {inp.type}"
        states[inp.name] = np.zeros(concrete_state_shape(inp), dtype=dtype)
    return states

def run_token(token_id, position, states):
    feeds = {
        "input_ids": np.asarray([[token_id]], dtype=np.int64),
        "attention_mask": np.ones((1, position + 1), dtype=np.int64),
        "position_ids": np.asarray([[position]], dtype=np.int64),
        **states,
    }
    expected = {x.name for x in session.get_inputs()}
    missing = expected - feeds.keys()
    assert not missing, f"Missing model inputs: {sorted(missing)}"
    started = time.perf_counter()
    values = session.run(None, {k: v for k, v in feeds.items() if k in expected})
    elapsed = time.perf_counter() - started
    outputs = dict(zip((x.name for x in session.get_outputs()), values, strict=True))
    logits = outputs["logits"]
    assert logits.shape[0] == 1 and logits.shape[1] == 1
    assert np.isfinite(logits).all(), "Non-finite logits detected"
    next_states = {}
    for name in states:
        suffix = name.removeprefix("past_key_values.")
        present_name = f"present.{suffix}"
        assert present_name in outputs, f"Missing output state: {present_name}"
        next_states[name] = outputs[present_name]
    top_id = int(np.argmax(logits[0, -1]))
    return next_states, top_id, elapsed, logits.shape

states = initial_states()
print("Initialized state tensors:", len(states))
states, top1, elapsed1, shape1 = run_token(1, 0, states)
states, top2, elapsed2, shape2 = run_token(2, 1, states)

print({
    "step_1": {"logits_shape": shape1, "top_token_id": top1, "seconds": elapsed1},
    "step_2": {"logits_shape": shape2, "top_token_id": top2, "seconds": elapsed2},
})
print("CPU stateful inference: PASS")


## 7. Hmm typed-decision API

This follows the tokenizer and autoregressive-cache flow from Mobius `examples/text_generation.py`, with two required adaptations for Qwen3.5 hybrid DeltaNet:

1. The prompt is prefetched one token at a time, feeding `present.*` back into `past_key_values.*` after each step.
2. Hmm does not generate normal text. It reads the first-token probabilities for `A/B/C...` after the prompt and normalizes them over the supplied options.

Every character in the prompt below, including the em dash `—`, is preserved from the official Hmm implementation.


In [ ]:
from transformers import AutoTokenizer
import math

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
MAX_OPTIONS = 255

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_REPO,
    revision=TOKENIZER_REVISION,
    trust_remote_code=False,
)
tokenizer.save_pretrained(OUTPUT_DIR)

letter_token_ids = {}
for letter in LETTERS:
    ids = tokenizer.encode(letter, add_special_tokens=False)
    assert len(ids) == 1, f"Option letter {letter!r} is not one token: {ids}"
    letter_token_ids[letter] = ids[0]

print("Tokenizer:", TOKENIZER_REPO, TOKENIZER_REVISION)
print("Option token IDs:", letter_token_ids)

def _text(value):
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))

def options_for(name, question):
    if not isinstance(question, dict) or question.get("instructions") is None:
        raise ValueError(f'question "{name}" needs "type" and "instructions"')

    question_type = question.get("type")
    criteria = question.get("criteria")
    if question_type == "choice":
        if not isinstance(criteria, dict) or not 2 <= len(criteria) <= MAX_OPTIONS:
            raise ValueError(f'choice "{name}" needs 2-{MAX_OPTIONS} options')
        return [(str(key), _text(value)) for key, value in criteria.items()]
    if question_type == "score":
        if not isinstance(criteria, list) or not 2 <= len(criteria) <= 10:
            raise ValueError(f'score "{name}" needs 2-10 ordered levels')
        return [(str(index), _text(value)) for index, value in enumerate(criteria)]
    if question_type == "noul":
        criteria = criteria if isinstance(criteria, dict) else {}
        return [
            ("false", _text(criteria.get("false", "No"))),
            ("true", _text(criteria.get("true", "Yes"))),
        ]
    raise ValueError(f'question "{name}" has unknown type {question_type!r}')

def build_hmm_prompt(state, question, options):
    lines = [f"{LETTERS[i]}: {key} — {description}" for i, (key, description) in enumerate(options)]
    user = (
        "State (data to evaluate):\n" + _text(state)
        + "\n\nQuestion:\n" + _text(question["instructions"])
        + "\n\nOptions:\n" + "\n".join(lines)
        + "\nReturn only the option letter."
    )
    return (
        f"<|im_start|>user\n{user}<|im_end|>\n"
        "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )


In [ ]:
output_names = [item.name for item in session.get_outputs()]
input_names = {item.name for item in session.get_inputs()}

def _forward_prompt(prompt):
    input_ids = tokenizer(prompt, add_special_tokens=False, return_tensors="np")[
        "input_ids"
    ].astype(np.int64)
    assert input_ids.shape[0] == 1 and input_ids.shape[1] > 0

    states = initial_states()
    outputs = None
    started = time.perf_counter()

    # Qwen3.5 linear-attention layers require token-by-token state threading.
    for position, token_id in enumerate(input_ids[0]):
        feeds = {
            "input_ids": np.asarray([[token_id]], dtype=np.int64),
            "attention_mask": np.ones((1, position + 1), dtype=np.int64),
            "position_ids": np.asarray([[position]], dtype=np.int64),
            **states,
        }
        missing = input_names - feeds.keys()
        assert not missing, f"Missing model inputs: {sorted(missing)}"
        values = session.run(output_names, {name: feeds[name] for name in input_names})
        outputs = dict(zip(output_names, values, strict=True))

        states = {
            name: outputs[f"present.{name.removeprefix('past_key_values.')}"]
            for name in states
        }

    assert outputs is not None
    logits = outputs["logits"][0, -1].astype(np.float64)
    assert np.isfinite(logits).all()
    return logits, int(input_ids.shape[1]), time.perf_counter() - started

def _letter_probabilities(prompt, letters):
    logits, input_tokens, elapsed = _forward_prompt(prompt)
    # Full-vocabulary softmax, matching llama.cpp/Ollama first-token probabilities.
    shifted = logits - np.max(logits)
    denominator = float(np.exp(shifted).sum())
    probabilities = {
        letter: float(np.exp(shifted[letter_token_ids[letter]]) / denominator)
        for letter in letters
    }
    return probabilities, input_tokens, elapsed

def _round4(value):
    return math.floor(value * 10000 + 0.5) / 10000

def answer_hmm_question(state, name, question):
    options = options_for(name, question)
    raw_probabilities = []
    input_tokens = 0
    elapsed = 0.0

    if len(options) <= len(LETTERS):
        letters = LETTERS[: len(options)]
        raw, count, seconds = _letter_probabilities(
            build_hmm_prompt(state, question, options), letters
        )
        raw_probabilities = [raw[letter] for letter in letters]
        input_tokens += count
        elapsed += seconds
    else:
        chunk_size = len(LETTERS) - 1
        none_option = ("none_of_these", "None of the other options fits")
        for start in range(0, len(options), chunk_size):
            chunk = options[start : start + chunk_size]
            chunk_options = [*chunk, none_option]
            letters = LETTERS[: len(chunk_options)]
            raw, count, seconds = _letter_probabilities(
                build_hmm_prompt(state, question, chunk_options), letters
            )
            raw_probabilities.extend(raw[LETTERS[i]] for i in range(len(chunk)))
            input_tokens += count
            elapsed += seconds

    total = sum(raw_probabilities)
    probabilities = (
        [value / total for value in raw_probabilities]
        if total > 0
        else [1.0 / len(options)] * len(options)
    )
    keys = [key for key, _ in options]
    best = int(np.argmax(probabilities))
    probability_map = {
        key: _round4(probabilities[index]) for index, key in enumerate(keys)
    }
    confidence = _round4((len(options) * probabilities[best] - 1) / (len(options) - 1))

    question_type = question["type"]
    if question_type == "noul":
        result = {"type": "noul", "noul": _round4(probabilities[1])}
    elif question_type == "choice":
        result = {
            "type": "choice",
            "choice": keys[best],
            "probabilities": probability_map,
            "confidence": confidence,
        }
    else:
        result = {
            "type": "score",
            "score": _round4(sum(index * p for index, p in enumerate(probabilities))),
            "legend": dict(options),
            "probabilities": probability_map,
            "confidence": confidence,
        }

    return {"input_tokens": input_tokens, "seconds": elapsed, "result": result}

def hmm_decide(body):
    if body.get("state") in (None, ""):
        raise ValueError('"state" is required')
    questions = body.get("questions")
    if not isinstance(questions, dict) or not questions:
        raise ValueError('"questions" must be a non-empty object')

    started = time.perf_counter()
    answers = {}
    input_tokens = 0
    inference_seconds = 0.0
    for name, question in questions.items():
        answer = answer_hmm_question(body["state"], name, question)
        answers[name] = answer["result"]
        input_tokens += answer["input_tokens"]
        inference_seconds += answer["seconds"]

    return {
        "model": GGUF_FILENAME,
        "answers": answers,
        "usage": {"input_tokens": input_tokens, "output_tokens": 0},
        "latency_ms": round((time.perf_counter() - started) * 1000),
        "inference_seconds": round(inference_seconds, 4),
    }


In [ ]:
request = {
    "state": "Help! My payouts have failed for 3 days. I need the money today.",
    "questions": {
        "is_urgent": {
            "type": "noul",
            "instructions": "Does this message convey urgency?",
        },
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "Payments, invoicing, refunds",
                "technical": "Bugs, outages, integrations",
                "sales": "Pricing, upgrades, new accounts",
            },
        },
        "priority": {
            "type": "score",
            "instructions": "How urgent is this request?",
            "criteria": ["Not urgent", "Normal", "Urgent", "Critical"],
        },
    },
}

hmm_result = hmm_decide(request)
print(json.dumps(hmm_result, ensure_ascii=False, indent=2))

assert hmm_result["answers"]["is_urgent"]["type"] == "noul"
assert 0.0 <= hmm_result["answers"]["is_urgent"]["noul"] <= 1.0
assert hmm_result["answers"]["is_urgent"]["noul"] >= 0.5, "Expected the sample to be classified as urgent"
assert hmm_result["answers"]["department"]["choice"] in request["questions"]["department"]["criteria"]
assert hmm_result["answers"]["department"]["choice"] == "billing", "Expected the payout failure to route to billing"
assert abs(sum(hmm_result["answers"]["department"]["probabilities"].values()) - 1.0) <= 0.001
assert 0.0 <= hmm_result["answers"]["priority"]["score"] <= 3.0
print("Hmm-compatible typed decision test: PASS")


## 8. Write the test summary and package the output


In [ ]:
summary = {
    "source": {
        "repo": HF_REPO,
        "revision": HF_REVISION,
        "filename": GGUF_FILENAME,
        "sha256": actual_sha,
        "architecture": architecture,
    },
    "converter": {
        "name": "onnxruntime/mobius",
        "source_revision": checked_out,
        "version": installed_mobius,
        "command": "mobius build-gguf <gguf> --output /content/onnx_outputs --ep cpu --dtype f32 --release",
    },
    "runtime": {
        "onnxruntime_version": installed_ort,
        "provider": session.get_providers()[0],
        "onnx_checker": "pass",
        "stateful_inference": "pass",
        "step_1_seconds": elapsed1,
        "step_2_seconds": elapsed2,
        "step_1_top_token_id": top1,
        "step_2_top_token_id": top2,
        "hmm_typed_decision": "pass",
    },
    "hmm_example": hmm_result,
    "note": "Mobius qwen35 real-weight runtime evidence is pending; this records direct CPU ORT acceptance for this exact artifact and environment.",
}
(OUTPUT_DIR / "cpu_test_summary.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(summary, indent=2))

archive = shutil.make_archive(str(WORKDIR / "onnx_outputs"), "zip", root_dir=OUTPUT_DIR)
print("Archive:", archive)
print("Final output directory:", OUTPUT_DIR)


Download `onnx_outputs.zip` from Colab's left-side **Files** panel. The complete uncompressed model remains in `/content/onnx_outputs/`.
